# Figure 2 & Figure 5 Replication (Safe Mode)

This notebook replicates Figure 2 (Prediction Performance) and Figure 5 (Control Performance). 
**Note:** PyBullet is set to DIRECT mode (headless) to prevent kernel crashes.

In [1]:
import sys
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import pybullet as p
import pybullet_data
import yaml
import time
import gc
from copy import deepcopy

# Add src to path
sys.path.append(os.path.abspath("src"))

# Import Project Modules
from koopman_model import DeepBilinearKoopman
from mpc_controller import BilinearMPC

# Load Configuration
with open('configs/default_config.yaml', 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
    
print("Configuration Loaded.")

Configuration Loaded.


## 1. Load Trained Model

In [2]:
model = DeepBilinearKoopman(cfg)
model_path = os.path.join(cfg['training']['checkpoint_dir'], 'best_model.pth')

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location='cpu'))
    print(f"Loaded model from {model_path}")
else:
    print(f"Warning: Model not found at {model_path}. Using random initialization.")
    
model.eval()

Loaded model from models\best_model.pth


DeepBilinearKoopman(
  (encoder): Sequential(
    (0): Linear(in_features=15, out_features=128, bias=True)
    (1): Tanh()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): Tanh()
    (4): Linear(in_features=128, out_features=128, bias=True)
    (5): Tanh()
    (6): Linear(in_features=128, out_features=20, bias=True)
  )
)

## 2. Setup Robot Environment (PyBullet)

In [5]:
def robust_load_urdf(urdf_path):
    """Helper to load URDF by fixing package paths for PyBullet"""
    # Ensure we use absolute path
    if not os.path.isabs(urdf_path):
         urdf_path = os.path.abspath(urdf_path)
    
    if not os.path.exists(urdf_path):
        raise FileNotFoundError(f"URDF file not found: {urdf_path}")
        
    with open(urdf_path, 'r', encoding='utf-8') as f:
        urdf_str = f.read()
        
    # Replace ROS package paths with relative paths
    new_urdf_str = urdf_str.replace("package://ur_description/meshes/ur5", "../meshes/ur5")
    
    # Save temporary fixed file with unique name to avoid conflicts
    temp_path = urdf_path.replace(".urdf", f"_temp_nb.urdf")
    with open(temp_path, 'w', encoding='utf-8') as f:
        f.write(new_urdf_str)
        
    return temp_path

def setup_pybullet(cfg, gui=False):
    # ALWAYS Disconnect first to be safe
    try:
        p.disconnect()
    except:
        pass
    
    # FORCE DIRECT MODE for stability in Notebooks
    # GUI mode in notebooks often causes 'Kernel Restarting' due to OpenGL context conflicts
    mode = p.DIRECT 
    print(f"Connecting to PyBullet in DIRECT mode...")
    p.connect(mode)
    p.resetSimulation()
    p.setAdditionalSearchPath(pybullet_data.getDataPath())
    p.setGravity(0, 0, -9.81)
    
    # Load Robot
    urdf_path = cfg['robot']['urdf_path']
    
    try:
        fixed_urdf = robust_load_urdf(urdf_path)
        robot_id = p.loadURDF(fixed_urdf, useFixedBase=True)
        # Cleanup temp file
        try:
            if os.path.exists(fixed_urdf):
                os.remove(fixed_urdf)
        except:
            pass
    except Exception as e:
        print(f"URDF Load Error: {e}")
        print("Retrying with raw path...")
        robot_id = p.loadURDF(urdf_path, useFixedBase=True)
        
    # Get Joint Info
    joint_indices = []
    num_joints = p.getNumJoints(robot_id)
    ee_idx = -1
    
    for i in range(num_joints):
        info = p.getJointInfo(robot_id, i)
        if info[2] == p.JOINT_REVOLUTE:
            joint_indices.append(i)
        
        # Check for EE link
        link_name = info[12].decode('utf-8')
        if cfg['robot']['ee_link_name'] == link_name:
            ee_idx = i
            
    if ee_idx == -1:
        ee_idx = joint_indices[-1] if joint_indices else 6
            
    return robot_id, joint_indices, ee_idx

robot_id, joint_indices, ee_idx = setup_pybullet(cfg, gui=False)
print(f"Robot Loaded. ID: {robot_id}, Joints: {len(joint_indices)}, EE Index: {ee_idx}")

Connecting to PyBullet in DIRECT mode...
URDF Load Error: Cannot load URDF file.
Retrying with raw path...


error: Cannot load URDF file.

## 3. Figure 2: Open-Loop Prediction Performance
We compare the ground truth trajectory (from PyBullet) vs. the model's predicted trajectory starting from the same initial state and applying the same control sequence.

In [ ]:
def collect_data(robot_id, joint_indices, ee_idx, steps=50):
    # Reset robot
    initial_q = np.random.uniform(-1, 1, 6)
    for i, idx in enumerate(joint_indices):
        p.resetJointState(robot_id, idx, initial_q[i])
        
    # Pre-allocate for performance
    gt_states = np.zeros((steps, 15))
    controls = np.zeros((steps, 6))
    
    # Constants
    dt = cfg['data']['time_step']
    p.setTimeStep(dt)
    
    print(f"Collecting data for {steps} steps...")
    
    for k in range(steps):
        # Measure x_k
        q = [p.getJointState(robot_id, i)[0] for i in joint_indices]
        dq = [p.getJointState(robot_id, i)[1] for i in joint_indices]
        
        # Get EE State
        # Note: In DIRECT mode, this is very fast
        link_state = p.getLinkState(robot_id, ee_idx)
        ee_pos = link_state[0]
        
        x_k = np.concatenate([q, dq, ee_pos])
        gt_states[k] = x_k
        
        # Random Control
        u_k = np.random.uniform(-0.3, 0.3, 6)
        controls[k] = u_k
        
        # Step
        p.setJointMotorControlArray(robot_id, joint_indices, p.VELOCITY_CONTROL, targetVelocities=u_k)
        p.stepSimulation()
        
        # Periodic print to show liveness
        if k % 10 == 0:
            # Minimal print to avoid I/O blocking
            sys.stdout.write('.') 
            
    print("\nDone.")
    return gt_states, controls

# Collect GT
gt_X, U_seq = collect_data(robot_id, joint_indices, ee_idx, steps=50)

# Predict with Model
pred_X = []
z_curr = model.get_full_state(torch.tensor(gt_X[0:1], dtype=torch.float32))

pred_X.append(gt_X[0]) # Initial state match

for k in range(len(U_seq)-1):
    u_in = torch.tensor(U_seq[k:k+1], dtype=torch.float32)
    with torch.no_grad():
        z_next = model.forward_step(z_curr, u_in)
    
    # Extract x from z (first 15 dims)
    x_next = z_next[0, :15].numpy()
    pred_X.append(x_next)
    z_curr = z_next

pred_X = np.array(pred_X)

# Plot Figure 2
plt.figure(figsize=(12, 5))
dims_to_plot = [0, 1, 2] # Plot first 3 joints
for i in dims_to_plot:
    plt.plot(gt_X[:, i], label=f'GT Joint {i}')
    plt.plot(pred_X[:, i], '--', label=f'Pred Joint {i}')
plt.title("Figure 2: Open-loop Prediction (3 Joints)")
plt.legend()
plt.grid()
plt.show()

## 4. Figure 5: Control Performance (Figure 8 Tracking)
We solve the MPC problem to track a Figure-8 trajectory.

In [ ]:
def get_figure8_trajectory(t): 
    # Using params from ref notebook
    a = 0.3
    x = 0.4 
    denom = 1 + np.sin(t)**2
    y = a * np.cos(t) / denom
    z = 0.5 + 2 * a * np.sin(t) * np.cos(t) / denom 
    return np.array([x, y, z])

def inverse_kinematics(ee_pos, robot_id, ee_idx):
    joint_pos = p.calculateInverseKinematics(robot_id, ee_idx, ee_pos)
    return np.array(joint_pos[:6])

# Setup Controller
mpc = BilinearMPC(model, cfg)

# Simulation Loop
steps = 200
time_seq = np.linspace(0, 10, steps)

actual_traj = []
desired_traj = []

# Reset to start
start_ee = get_figure8_trajectory(0)
init_q = inverse_kinematics(start_ee, robot_id, ee_idx)
for i, idx in enumerate(joint_indices):
    p.resetJointState(robot_id, idx, init_q[i])

current_q = init_q
current_dq = np.zeros(6)

print("Starting MPC Control Loop...")
for k in range(steps):
    t = time_seq[k]
    
    # 1. State Estimation
    link_state = p.getLinkState(robot_id, ee_idx)
    current_ee = np.array(link_state[0])
    x_k = np.concatenate([current_q, current_dq, current_ee])
    
    actual_traj.append(current_ee)
    
    # 2. Reference
    ref_ee = get_figure8_trajectory(t)
    ref_q = inverse_kinematics(ref_ee, robot_id, ee_idx)
    x_ref = np.concatenate([ref_q, np.zeros(6), ref_ee])
    
    desired_traj.append(ref_ee)
    
    # Data Prep
    z_k = model.get_full_state(torch.tensor(x_k[None, :], dtype=torch.float32)).detach().numpy()[0]
    z_ref = model.get_full_state(torch.tensor(x_ref[None, :], dtype=torch.float32)).detach().numpy()[0]
    
    # 3. Solve MPC
    mpc.update_model_matrices(z_k)
    u_opt = mpc.solve(z_k, z_ref)
    
    # 4. Apply
    p.setJointMotorControlArray(robot_id, joint_indices, p.VELOCITY_CONTROL, targetVelocities=u_opt)
    p.stepSimulation()
    
    # Update
    current_q = np.array([p.getJointState(robot_id, i)[0] for i in joint_indices])
    current_dq = np.array([p.getJointState(robot_id, i)[1] for i in joint_indices])
    
    if k % 10 == 0:
        sys.stdout.write('.')

actual_traj = np.array(actual_traj)
desired_traj = np.array(desired_traj)

# Plot Figure 5
plt.figure(figsize=(8, 8))
plt.plot(desired_traj[:, 1], desired_traj[:, 2], 'g--', label='Desired (Eq 27)')
plt.plot(actual_traj[:, 1], actual_traj[:, 2], 'b-', label='Actual (DBK-MPC)')
plt.xlabel("Y (m)")
plt.ylabel("Z (m)")
plt.title("Figure 5: Trajectory Tracking of Figure-8")
plt.legend()
plt.axis('equal')
plt.grid()
plt.show()